# BASIC Code Generation with Chain-of-Thought Reasoning

From scratch, train a GPT model that:
1. Takes an English prompt (single-digit arithmetic)
2. Outputs a natural language reasoning trace (CoT)
3. Outputs a BASIC program

Output format: `<prompt> <sep> <reasoning> <code> <basic_code> <eos>`

## 1. Configuration

In [ ]:
import os

class Config:
    data_dir = "./data"
    data_path = os.path.join(data_dir, "dataset.json")
    tokenizer_path = os.path.join(data_dir, "tokenizer.json")
    model_dir = "./checkpoints"
    model_path = os.path.join(model_dir, "model.pt")

    num_samples = 50000
    train_ratio = 0.95
    seed = 42

    vocab_size = 2048
    max_seq_len = 256

    d_model = 256
    n_heads = 8
    n_layers = 6
    d_ff = 1024
    dropout = 0.1

    batch_size = 64
    learning_rate = 3e-4
    weight_decay = 0.01
    max_epochs = 30
    warmup_steps = 500
    grad_clip = 1.0

    num_workers = 2
    num_gpus = 2

    inference_max_len = 256
    inference_temperature = 0.8
    inference_top_k = 50

    special_tokens = ["<pad>", "<eos>", "<sep>", "<unk>", "<code>"]
    pad_id = 0
    eos_id = 1
    sep_id = 2
    unk_id = 3
    code_id = 4

config = Config()
print("Configuration loaded.")
print(f"  max_seq_len: {config.max_seq_len}")
print(f"  special_tokens: {config.special_tokens}")

## 2. Dataset Generation (with CoT Reasoning)

In [ ]:
import json
import random
import gc

ADD_TEMPLATES = [
    "Input {a} and {b}, and print their sum",
    "Calculate the sum of {a} and {b}",
    "Add {a} and {b} together and print the result",
    "Write a program to add {a} and {b}",
    "Find the sum of {a} and {b}",
    "Compute {a} plus {b}",
    "What is {a} added to {b}?",
    "Print the result of adding {a} and {b}",
    "Add together {a} and {b} and display it",
    "Show the sum when you add {a} and {b}",
    "Calculate {a} + {b}",
    "Find the total of {a} and {b}",
    "Sum up {a} and {b} and print",
    "Get the addition result of {a} and {b}",
    "What do you get when you add {a} and {b}?",
]

SUB_TEMPLATES = [
    "Input {a} and {b}, and print their difference",
    "Calculate the difference of {a} and {b}",
    "Subtract {b} from {a} and print the result",
    "Write a program to subtract {b} from {a}",
    "Find the difference between {a} and {b}",
    "Compute {a} minus {b}",
    "What is {a} subtracted by {b}?",
    "Print the result of subtracting {b} from {a}",
    "Show the difference when you subtract {b} from {a}",
    "Calculate {a} - {b}",
    "Subtract {b} from {a} and display the answer",
    "What is the result of {a} minus {b}?",
    "Find {a} take away {b}",
    "Get the subtraction result of {a} and {b}",
    "What do you get when you subtract {b} from {a}?",
]

MUL_TEMPLATES = [
    "Input {a} and {b}, and print their product",
    "Calculate the product of {a} and {b}",
    "Multiply {a} and {b} together and print the result",
    "Write a program to multiply {a} and {b}",
    "Find the product of {a} and {b}",
    "Compute {a} times {b}",
    "What is {a} multiplied by {b}?",
    "Print the result of multiplying {a} and {b}",
    "Show the product when you multiply {a} and {b}",
    "Calculate {a} * {b}",
    "Multiply {a} by {b} and display the answer",
    "What is the result of {a} times {b}?",
    "Find the multiplication of {a} and {b}",
    "Get the product of {a} and {b}",
    "What do you get when you multiply {a} and {b}?",
]

DIV_TEMPLATES = [
    "Input {a} and {b}, and print their quotient",
    "Calculate the quotient of {a} and {b}",
    "Divide {a} by {b} and print the result",
    "Write a program to divide {a} by {b}",
    "Find the quotient of {a} divided by {b}",
    "Compute {a} divided by {b}",
    "What is {a} divided by {b}?",
    "Print the result of dividing {a} by {b}",
    "Show the quotient when you divide {a} by {b}",
    "Calculate {a} / {b}",
    "Divide {a} by {b} and display the answer",
    "What is the result of {a} divided by {b}?",
    "Find the division of {a} by {b}",
    "Get the quotient of {a} and {b}",
    "What do you get when you divide {a} by {b}?",
]

ADD_REASONING = [
    "The user wants to add {a} and {b}. I need to create a BASIC program that assigns {a} to A and {b} to B, then prints their sum. I will use assignment statements to set A={a} and B={b}, then use PRINT with the + operator to display the result.",
    "This is an addition problem with {a} and {b}. The program should assign A={a} and B={b}, then output their sum using PRINT A + B.",
    "To add {a} and {b}, I need a BASIC program with assignment statements A={a} and B={b}, and a PRINT statement that adds them with the + operator.",
    "The task is to compute the sum of {a} and {b}. I will write a BASIC program that assigns A={a} and B={b}, then prints A + B.",
    "For adding {a} and {b}, the program needs to assign A={a} and B={b}, then display their sum using PRINT A + B.",
]

SUB_REASONING = [
    "The user wants to subtract {b} from {a}. I need to create a BASIC program that assigns {a} to A and {b} to B, then prints their difference. I will use assignment statements to set A={a} and B={b}, then use PRINT with the - operator to display the result.",
    "This is a subtraction problem with {a} and {b}. The program should assign A={a} and B={b}, then output their difference using PRINT A - B.",
    "To subtract {b} from {a}, I need a BASIC program with assignment statements A={a} and B={b}, and a PRINT statement that subtracts them with the - operator.",
    "The task is to compute the difference of {a} and {b}. I will write a BASIC program that assigns A={a} and B={b}, then prints A - B.",
    "For subtracting {b} from {a}, the program needs to assign A={a} and B={b}, then display their difference using PRINT A - B.",
]

MUL_REASONING = [
    "The user wants to multiply {a} and {b}. I need to create a BASIC program that assigns {a} to A and {b} to B, then prints their product. I will use assignment statements to set A={a} and B={b}, then use PRINT with the * operator to display the result.",
    "This is a multiplication problem with {a} and {b}. The program should assign A={a} and B={b}, then output their product using PRINT A * B.",
    "To multiply {a} and {b}, I need a BASIC program with assignment statements A={a} and B={b}, and a PRINT statement that multiplies them with the * operator.",
    "The task is to compute the product of {a} and {b}. I will write a BASIC program that assigns A={a} and B={b}, then prints A * B.",
    "For multiplying {a} and {b}, the program needs to assign A={a} and B={b}, then display their product using PRINT A * B.",
]

DIV_REASONING = [
    "The user wants to divide {a} by {b}. I need to create a BASIC program that assigns {a} to A and {b} to B, then prints their quotient. I will use assignment statements to set A={a} and B={b}, then use PRINT with the / operator to display the result.",
    "This is a division problem with {a} and {b}. The program should assign A={a} and B={b}, then output their quotient using PRINT A / B.",
    "To divide {a} by {b}, I need a BASIC program with assignment statements A={a} and B={b}, and a PRINT statement that divides them with the / operator.",
    "The task is to compute the quotient of {a} divided by {b}. I will write a BASIC program that assigns A={a} and B={b}, then prints A / B.",
    "For dividing {a} by {b}, the program needs to assign A={a} and B={b}, then display their quotient using PRINT A / B.",
]

def generate_basic_add(a, b):
    return f"10 A = {a}\n20 B = {b}\n30 PRINT A + B"

def generate_basic_sub(a, b):
    return f"10 A = {a}\n20 B = {b}\n30 PRINT A - B"

def generate_basic_mul(a, b):
    return f"10 A = {a}\n20 B = {b}\n30 PRINT A * B"

def generate_basic_div(a, b):
    return f"10 A = {a}\n20 B = {b}\n30 PRINT A / B"

def generate_dataset(num_samples, seed=42):
    random.seed(seed)
    dataset = []
    ops = [
        ("add", ADD_TEMPLATES, ADD_REASONING, generate_basic_add),
        ("sub", SUB_TEMPLATES, SUB_REASONING, generate_basic_sub),
        ("mul", MUL_TEMPLATES, MUL_REASONING, generate_basic_mul),
        ("div", DIV_TEMPLATES, DIV_REASONING, generate_basic_div),
    ]
    per_op = num_samples // 4
    for op_name, templates, reasoning_templates, gen_func in ops:
        for _ in range(per_op):
            a = random.randint(0, 9)
            b = random.randint(0, 9)
            if op_name == "div" and b == 0:
                b = random.randint(1, 9)
            template = random.choice(templates)
            prompt = template.format(a=a, b=b)
            reasoning_template = random.choice(reasoning_templates)
            reasoning = reasoning_template.format(a=a, b=b)
            basic_code = gen_func(a, b)
            dataset.append({
                "prompt": prompt,
                "reasoning": reasoning,
                "code": basic_code,
                "operation": op_name,
                "a": a,
                "b": b,
            })
    random.shuffle(dataset)
    return dataset

os.makedirs(config.data_dir, exist_ok=True)
dataset = generate_dataset(config.num_samples, config.seed)
with open(config.data_path, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)

print(f"Generated {len(dataset)} samples")
print(f"\nSample data:")
for i in range(3):
    print(f"\n  [{i}] Prompt:    {dataset[i]['prompt']}")
    print(f"      Reasoning: {dataset[i]['reasoning']}")
    print(f"      Code:      {repr(dataset[i]['code'])}")

## 3. Tokenizer (Character-Level)

In [ ]:
class CharTokenizer:
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
        self.special_tokens = config.special_tokens
        self.pad_id = config.pad_id
        self.eos_id = config.eos_id
        self.sep_id = config.sep_id
        self.unk_id = config.unk_id
        self.code_id = config.code_id
        for i, token in enumerate(self.special_tokens):
            self.char_to_id[token] = i
            self.id_to_char[i] = token

    def train(self, texts):
        chars = set()
        for text in texts:
            chars.update(set(text))
        chars = sorted(chars)
        idx = len(self.special_tokens)
        for char in chars:
            if char not in self.char_to_id:
                self.char_to_id[char] = idx
                self.id_to_char[idx] = char
                idx += 1

    def encode(self, text):
        ids = []
        for char in text:
            if char in self.char_to_id:
                ids.append(self.char_to_id[char])
            else:
                ids.append(self.unk_id)
        return ids

    def decode(self, ids):
        chars = []
        for id_ in ids:
            if id_ in self.id_to_char:
                token = self.id_to_char[id_]
                if token == "<eos>":
                    break
                if token == "<pad>":
                    continue
                if token in self.special_tokens:
                    chars.append(" ")
                    continue
                chars.append(token)
            else:
                chars.append("?")
        return "".join(chars)

    def save(self, path):
        data = {
            "char_to_id": self.char_to_id,
            "special_tokens": self.special_tokens,
        }
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.char_to_id = {k: int(v) for k, v in data["char_to_id"].items()}
        self.id_to_char = {int(v): k for k, v in self.char_to_id.items()}
        self.special_tokens = data["special_tokens"]

    @property
    def get_vocab_size(self):
        return len(self.char_to_id)

all_texts = []
for item in dataset:
    all_texts.append(item["prompt"])
    all_texts.append(item["reasoning"])
    all_texts.append(item["code"])

tokenizer = CharTokenizer()
tokenizer.train(all_texts)
tokenizer.save(config.tokenizer_path)

del all_texts
gc.collect()

print(f"Vocab size: {tokenizer.get_vocab_size}")
print(f"Special tokens: {tokenizer.special_tokens}")

test_text = "10 INPUT A\n20 PRINT A + B"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(f"\nTokenizer test:")
print(f"  Original: {repr(test_text)}")
print(f"  Encoded:  {encoded[:20]}...")
print(f"  Decoded:  {repr(decoded)}")

## 4. Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class BasicDataset(Dataset):
    def __init__(self, data, tokenizer, max_seq_len, is_train=True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.is_train = is_train

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item["prompt"]
        reasoning = item["reasoning"]
        code = item["code"]

        prompt_ids = self.tokenizer.encode(prompt)
        reasoning_ids = self.tokenizer.encode(reasoning)
        code_ids = self.tokenizer.encode(code)

        input_ids = (
            prompt_ids
            + [self.tokenizer.sep_id]
            + reasoning_ids
            + [self.tokenizer.code_id]
            + code_ids
            + [self.tokenizer.eos_id]
        )

        sep_pos = len(prompt_ids)
        labels = [-100] * (sep_pos + 1) + input_ids[sep_pos + 1:]

        if len(input_ids) > self.max_seq_len:
            input_ids = input_ids[:self.max_seq_len]
            labels = labels[:self.max_seq_len]

        pad_len = self.max_seq_len - len(input_ids)
        input_ids = input_ids + [self.tokenizer.pad_id] * pad_len
        labels = labels + [-100] * pad_len

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

split_idx = int(len(dataset) * config.train_ratio)
train_data = dataset[:split_idx]
val_data = dataset[split_idx:]

train_dataset = BasicDataset(train_data, tokenizer, config.max_seq_len, is_train=True)
val_dataset = BasicDataset(val_data, tokenizer, config.max_seq_len, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config.batch_size,
    shuffle=False,
    num_workers=config.num_workers,
    pin_memory=True,
)

print(f"Train: {len(train_data)}, Val: {len(val_data)}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

sample = train_dataset[0]
print(f"\nSample batch:")
print(f"  input_ids shape: {sample['input_ids'].shape}")
print(f"  labels shape:    {sample['labels'].shape}")
decoded_input = tokenizer.decode(sample['input_ids'].tolist())
print(f"  Decoded input:   {repr(decoded_input[:120])}...")

## 5. GPT Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_model = d_model
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        attn = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            attn = attn.masked_fill(~mask, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.attn_dropout(attn)
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        x = x + self.attn(self.ln1(x), mask)
        x = x + self.ff(self.ln2(x))
        return x

class GPTModel(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        if config is None:
            config = Config
        self.config = config
        self.pad_id = config.pad_id
        self.eos_id = config.eos_id
        self.sep_id = config.sep_id
        self.code_id = config.code_id
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_id)
        self.pos_encoding = PositionalEncoding(config.d_model, config.max_seq_len, config.dropout)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff, config.dropout)
            for _ in range(config.n_layers)
        ])
        self.ln_f = nn.LayerNorm(config.d_model)
        self.head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.token_embedding.weight = self.head.weight
        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"Model parameters: {n_params / 1e6:.2f}M")

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.padding_idx is not None:
                nn.init.zeros_(module.weight[module.padding_idx])
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, input_ids, labels=None):
        B, T = input_ids.size()
        assert T <= self.config.max_seq_len, f"Sequence length {T} exceeds max {self.config.max_seq_len}"
        causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=input_ids.device)).unsqueeze(0).unsqueeze(0)
        pad_mask = (input_ids != self.pad_id).unsqueeze(1).unsqueeze(2).expand(-1, 1, T, -1)
        mask = causal_mask & pad_mask
        tok_emb = self.token_embedding(input_ids)
        x = self.pos_encoding(tok_emb)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            ).unsqueeze(0)
        return logits, loss

    @torch.no_grad()
    def generate(self, input_ids, max_len=256, temperature=0.8, top_k=50):
        self.eval()
        device = next(self.parameters()).device
        input_ids = input_ids.to(device)
        for _ in range(max_len):
            idx_cond = input_ids if input_ids.size(1) <= self.config.max_seq_len else input_ids[:, -self.config.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_id], dim=1)
            if next_id.item() == self.eos_id:
                break
        return input_ids

model = GPTModel(config)
print(f"Model created on {next(model.parameters()).device}")

## 6. Training

In [ ]:
import gc
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
n_gpus = torch.cuda.device_count()
print(f"Device: {device}, GPUs: {n_gpus}")

if n_gpus > 1:
    model = nn.DataParallel(model)
model = model.to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
    betas=(0.9, 0.95),
)

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step > max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

total_steps = len(train_loader) * config.max_epochs
warmup_steps = config.warmup_steps

print(f"Total steps: {total_steps}, Warmup steps: {warmup_steps}")
print(f"Starting training for {config.max_epochs} epochs...")
print("=" * 70)

best_val_loss = float("inf")
os.makedirs(config.model_dir, exist_ok=True)

for epoch in range(config.max_epochs):
    model.train()
    total_loss = 0.0
    n_batches = 0
    start_time = time.time()

    for batch_idx, batch in enumerate(train_loader):
        step = epoch * len(train_loader) + batch_idx
        lr = get_lr(step, warmup_steps, total_steps, config.learning_rate, config.learning_rate * 0.1)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits, loss = model(input_ids, labels=labels)
        if n_gpus > 1:
            loss = loss.mean()

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / n_batches
            print(f"  Epoch {epoch+1}/{config.max_epochs} | Step {batch_idx+1}/{len(train_loader)} | Loss: {avg_loss:.4f} | LR: {lr:.6f}")

    avg_train_loss = total_loss / n_batches
    epoch_time = time.time() - start_time

    model.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            logits, loss = model(input_ids, labels=labels)
            if n_gpus > 1:
                loss = loss.mean()
            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches

    print(f"Epoch {epoch+1}/{config.max_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Time: {epoch_time:.1f}s")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_model = model.module if n_gpus > 1 else model
        torch.save({
            "epoch": epoch,
            "model_state_dict": save_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": avg_val_loss,
        }, config.model_path)
        print(f"  -> Best model saved (val_loss: {avg_val_loss:.4f})")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 70)
print(f"Training complete! Best val loss: {best_val_loss:.4f}")

## 7. Inference

In [ ]:
inference_model = GPTModel(config)
checkpoint = torch.load(config.model_path, map_location=device, weights_only=False)
inference_model.load_state_dict(checkpoint["model_state_dict"])
inference_model = inference_model.to(device)
inference_model.eval()
print(f"Model loaded from {config.model_path}")
print(f"Best val loss: {checkpoint['val_loss']:.4f}")

In [ ]:
def generate_basic(model, tokenizer, prompt, device,
                   max_len=256, temperature=0.8, top_k=50):
    prompt_ids = tokenizer.encode(prompt)
    input_ids = prompt_ids + [tokenizer.sep_id]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    output_ids = model.generate(
        input_tensor,
        max_len=max_len,
        temperature=temperature,
        top_k=top_k,
    )

    generated = output_ids[0].tolist()

    sep_pos = None
    code_sep_pos = None
    for i, id_ in enumerate(generated):
        if id_ == tokenizer.sep_id and sep_pos is None:
            sep_pos = i
        if id_ == tokenizer.code_id and sep_pos is not None and code_sep_pos is None:
            code_sep_pos = i

    if sep_pos is not None and code_sep_pos is not None:
        reasoning_ids = generated[sep_pos + 1: code_sep_pos]
        code_ids = generated[code_sep_pos + 1:]
    elif sep_pos is not None:
        reasoning_ids = []
        code_ids = generated[sep_pos + 1:]
    else:
        reasoning_ids = []
        code_ids = generated[len(prompt_ids):]

    reasoning_text = tokenizer.decode(reasoning_ids)
    code_text = tokenizer.decode(code_ids)

    print(f"[Prompt]    {prompt}")
    print(f"[Reasoning] {reasoning_text.strip()}")
    print(f"[BASIC Code]")
    for line in code_text.strip().split(chr(10)):
        print(f"  {line}")

    return reasoning_text, code_text

print("Inference function ready.")

## 8. Test Results

In [ ]:
test_prompts = [
    "Input 3 and 4, and print their sum",
    "Calculate the difference of 9 and 2",
    "Multiply 5 and 6 together and print the result",
    "Divide 8 by 4 and print the result",
    "Find the sum of 7 and 1",
    "Subtract 3 from 8 and print the result",
    "What is 6 multiplied by 7?",
    "Compute 9 divided by 3",
    "Add 2 and 5 together and print the result",
    "What do you get when you subtract 4 from 7?",
]

print("=" * 70)
print("BASIC Code Generation with CoT Reasoning")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    print(f"\n{'─' * 70}")
    print(f"Test #{i + 1}")
    reasoning, code = generate_basic(
        inference_model, tokenizer, prompt, device,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    print("─" * 70)

## 9. Accuracy Test

In [ ]:
correct = 0
total = len(test_prompts)
expected_ops = ["+", "-", "*", "/", "+", "-", "*", "/", "+", "-"]

print("Accuracy Test")
print("=" * 70)

for i, prompt in enumerate(test_prompts):
    prompt_ids = tokenizer.encode(prompt)
    input_ids = prompt_ids + [tokenizer.sep_id]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    output_ids = inference_model.generate(
        input_tensor,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    generated = output_ids[0].tolist()

    code_sep_pos = None
    for j, id_ in enumerate(generated):
        if id_ == tokenizer.code_id:
            code_sep_pos = j
            break

    if code_sep_pos is not None:
        code_text = tokenizer.decode(generated[code_sep_pos + 1:])
    else:
        code_text = tokenizer.decode(generated[len(prompt_ids):])

    has_assign = "=" in code_text
    has_print = "PRINT" in code_text
    has_op = expected_ops[i] in code_text
    is_correct = has_assign and has_print and has_op
    correct += is_correct
    status = "PASS" if is_correct else "FAIL"
    print(f"[{status}] {prompt}")
    if not is_correct:
        print(f"       Generated: {repr(code_text)}")
        print(f"       Has =: {has_assign}, Has PRINT: {has_print}, Has '{expected_ops[i]}': {has_op}")

print(f"\nAccuracy: {correct}/{total} ({100*correct/total:.1f}%)")

## 10. Custom Test

In [ ]:
custom_prompts = [
    "Input 5 and 3, and print their sum",
    "Calculate the product of 4 and 7",
    "What is 8 divided by 2?",
]

for i, prompt in enumerate(custom_prompts):
    print(f"\n{'─' * 70}")
    print(f"Custom Test #{i + 1}")
    reasoning, code = generate_basic(
        inference_model, tokenizer, prompt, device,
        max_len=config.inference_max_len,
        temperature=0.1,
        top_k=config.inference_top_k,
    )
    print("─" * 70)

## 11. Cleanup

In [ ]:
import shutil

if os.path.exists(config.data_dir):
    shutil.rmtree(config.data_dir)
    print(f"Cleaned up {config.data_dir}")

if os.path.exists(config.model_dir):
    shutil.rmtree(config.model_dir)
    print(f"Cleaned up {config.model_dir}")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nAll done! Memory cleaned up.")